In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/Phenikaa_Campus_Courier_2026.zip" -d "/content/dataset"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/Phenikaa_Campus_Courier_2026.zip
   creating: /content/dataset/Phenikaa_Campus_Courier_2026/
  inflating: /content/dataset/Phenikaa_Campus_Courier_2026/DE_BAI.md  
   creating: /content/dataset/Phenikaa_Campus_Courier_2026/starter/
  inflating: /content/dataset/Phenikaa_Campus_Courier_2026/starter/starter.py  
   creating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/
   creating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/test/
   creating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/test/images/
  inflating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/test/images/3cabcd4d85984f57.png  
  inflating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/test/images/c9b768991f386ca4.png  
  inflating: /content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/test/images/e9543082798b7d67.png  
  inflating: /content/dataset/Phenikaa_Campus_Courier_

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 10.8 MB/s eta 0:00:00


In [ ]:
import os, json, cv2, yaml
from tqdm.notebook import tqdm

# ==========================================
# CẤU HÌNH
# ==========================================
ROOT        = "/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public"
OUT_BASE    = "/content/YOLO_Full_Dataset"
TARGET_SIZE = 1024
BOX_PAD     = 15          # đệm quanh cạnh (px, toạ độ ảnh gốc)

CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered'
]
CLASS_MAP     = {n: i for i, n in enumerate(CLASSES)}
VISUAL_TO_IDX = {'normal': 1, 'crowded': 2, 'covered': 3}


def to_yolo(xmin, ymin, xmax, ymax, w, h):
    xc, yc = ((xmin + xmax) / 2) / w, ((ymin + ymax) / 2) / h
    bw, bh = (xmax - xmin) / w, (ymax - ymin) / h
    c = lambda v: max(0.0, min(1.0, v))
    return c(xc), c(yc), c(bw), c(bh)


def oneway_class(edge, rc1, rc2):
    r_dst, c_dst = edge['oneway_to']
    r_src, c_src = rc1 if tuple(edge['oneway_to']) == rc2 else rc2
    if c_dst > c_src: return 'oneway_right'
    if c_dst < c_src: return 'oneway_left'
    if r_dst > r_src: return 'oneway_down'
    if r_dst < r_src: return 'oneway_up'
    return None


def scene_to_lines(scene, w_orig, h_orig):
    """1 scene -> list dòng nhãn YOLO.
    THAY ĐỔI CHÍNH: mỗi cạnh có icon được ghi THÀNH 2 BOX CÙNG TOẠ ĐỘ:
        (1) box kiểu nét  (edge_type_k / print_edge_k)  -> model học status
        (2) box icon      (stairs / oneway_*)            -> model học icon
    Cạnh 'closed' vẫn là 1 box 'closed' (closed không đi kèm stairs/oneway)."""
    is_print  = (scene.get('style') == 'print')
    prefix    = 'print_edge' if is_print else 'edge_type'
    road_look = scene.get('road_look', {})
    lines = []

    def add(cls_name, bbox):
        if cls_name is None or cls_name not in CLASS_MAP: return
        xc, yc, w, h = to_yolo(*bbox, w_orig, h_orig)
        lines.append(f"{CLASS_MAP[cls_name]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    node_xy = {tuple(n['rc']): n['xy'] for n in scene['nodes']}

    # ---- CẠNH
    for e in scene.get('edges', []):
        rc1, rc2 = tuple(e['a']), tuple(e['b'])
        if rc1 not in node_xy or rc2 not in node_xy: continue
        (x1, y1), (x2, y2) = node_xy[rc1], node_xy[rc2]
        box = [min(x1, x2) - BOX_PAD, min(y1, y2) - BOX_PAD,
               max(x1, x2) + BOX_PAD, max(y1, y2) + BOX_PAD]
        status = e.get('status')

        if status == 'closed':
            add('closed', box)
            continue

        # (1) box kiểu nét: LUÔN có, kể cả cạnh có icon. road_look = tráo đổi nét <-> status theo từng scene
        look = VISUAL_TO_IDX.get(road_look.get(status, status))
        if look is not None:
            add(f"{prefix}_{look}", box)

        # (2) box icon cùng toạ độ
        if e.get('oneway_to') is not None:
            add(oneway_class(e, rc1, rc2), box)
        if e.get('stairs'):
            add('stairs', box)

    # ---- ĐỊA ĐIỂM
    for lm in scene.get('landmarks', []):
        rc = tuple(lm['rc'])
        if rc in node_xy:
            x, y = node_xy[rc]
            add(lm['type'], [x - 20, y - 20, x + 20, y + 20])

    # ---- ROBOT
    if 'robot' in scene and tuple(scene['robot']['rc']) in node_xy:
        x, y = node_xy[tuple(scene['robot']['rc'])]
        add(f"robot_{scene['robot']['heading'].lower()}", [x - 20, y - 20, x + 20, y + 20])

    # ---- THỜI TIẾT
    if scene.get('weather_box'):
        add('weather_sun' if scene.get('weather', 'dry') == 'dry' else 'weather_rain', scene['weather_box'])

    # ---- CHÚ GIẢI
    for leg in scene.get('legend', []):
        kind = leg.get('kind')
        if kind in ('normal', 'crowded', 'covered'):
            idx = VISUAL_TO_IDX.get(road_look.get(kind, kind), 1)
            if leg.get('swatch'):
                add(f"swatch_{'print' if is_print else 'edge'}_{idx}", leg['swatch'])
            if leg.get('label'):
                add(f"txt_{kind}", leg['label'])
    return lines


def build_split(split):
    """split = 'train' hoặc 'validation' -> OUT_BASE/images|labels/{train|val}"""
    tag = 'train' if split == 'train' else 'val'
    base = os.path.join(ROOT, split)
    img_dir = os.path.join(OUT_BASE, 'images', tag)
    lbl_dir = os.path.join(OUT_BASE, 'labels', tag)
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)

    scenes = json.load(open(os.path.join(base, 'scenes.json'), encoding='utf-8'))
    for sc in tqdm(scenes, desc=f"YOLO labels [{tag}]"):
        img = cv2.imread(os.path.join(base, 'images', os.path.basename(sc['image'])))
        if img is None: continue
        h0, w0 = img.shape[:2]
        cv2.imwrite(os.path.join(img_dir, f"{sc['scene_id']}.jpg"),
                    cv2.resize(img, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA))
        with open(os.path.join(lbl_dir, f"{sc['scene_id']}.txt"), 'w') as f:
            f.write('\n'.join(scene_to_lines(sc, w0, h0)))


build_split('train')
build_split('validation')

# data.yaml: val = tập validation THẬT (không trỏ về train)
with open(os.path.join(OUT_BASE, 'data.yaml'), 'w', encoding='utf-8') as f:
    yaml.dump({'train': f"{OUT_BASE}/images/train", 'val': f"{OUT_BASE}/images/val",
               'nc': len(CLASSES), 'names': CLASSES}, f, sort_keys=False)

# xoá cache nhãn cũ (nếu không YOLO sẽ đọc lại nhãn 1 box)
import glob
for c in glob.glob(f"{OUT_BASE}/labels/*.cache"): os.remove(c)
print("Xong. Nhãn cạnh có icon giờ là 2 box (kiểu nét + icon).")

YOLO labels [train]:   0%|          | 0/2000 [00:00<?, ?it/s]

YOLO labels [val]:   0%|          | 0/300 [00:00<?, ?it/s]

Xong. Nhãn cạnh có icon giờ là 2 box (kiểu nét + icon).


In [ ]:
# =====================================================================
# TẠO YOLO_Aug: CÂN BẰNG TỔ HỢP (kiểu nét x icon) BẰNG NHÂN BẢN ẢNH
# - Dựa trên nhãn 2 box mới (cạnh có icon = box kiểu nét + box icon cùng toạ độ)
# - Không dán landmark (landmark vốn đã cân), không khai thác ca khó trên chính tập train
# - val = tập validation THẬT
# =====================================================================
import os, glob, shutil, yaml
from collections import Counter, defaultdict

BASE = "/content/YOLO_Full_Dataset"
OUT  = "/content/YOLO_Aug"
SRC_IMG, SRC_LBL = f"{BASE}/images/train", f"{BASE}/labels/train"
VAL_IMG, VAL_LBL = f"{BASE}/images/val",   f"{BASE}/labels/val"

MAX_COPIES = 4      # tối đa số bản nhân thêm cho 1 ảnh
MIN_RATIO  = 1.5    # tổ hợp phải hiếm hơn tổ hợp phổ biến nhất >=1.5 lần mới nhân

CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3', 'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered']
LOOK   = set(range(14, 20))
STAIRS = 21
ONEWAY = set(range(22, 26))


def link(src, dst):
    if os.path.exists(dst): return
    try: os.link(src, dst)
    except Exception: shutil.copy(src, dst)


def img_path(i, folder):
    for ext in ('.jpg', '.png'):
        if os.path.exists(f"{folder}/{i}{ext}"): return f"{folder}/{i}{ext}"
    return None


def edge_combos(label_path):
    """Đếm tổ hợp (class kiểu nét, loại icon) của các cạnh có icon trong 1 ảnh.
    Box cùng toạ độ = cùng 1 cạnh."""
    grp = defaultdict(set)
    for l in open(label_path, encoding='utf-8'):
        p = l.split()
        if len(p) < 5: continue
        grp[tuple(round(float(v), 4) for v in p[1:5])].add(int(float(p[0])))
    out = Counter()
    for cs in grp.values():
        look = [c for c in cs if c in LOOK]
        if not look: continue
        if STAIRS in cs:  out[(look[0], 'stairs')] += 1
        if cs & ONEWAY:   out[(look[0], 'oneway')] += 1
    return out


ids = sorted({os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{SRC_IMG}/*.*")})
per_img = {i: edge_combos(f"{SRC_LBL}/{i}.txt") for i in ids if os.path.exists(f"{SRC_LBL}/{i}.txt")}

total = Counter()
for c in per_img.values(): total += c
print("Tổ hợp (kiểu nét x icon) trong train:")
for (k, m), n in sorted(total.items()):
    print(f"  {CLASSES[k]:<14}+ {m:<7}{n:>6}")

top = max(total.values()) if total else 1
copies = {}
for i, c in per_img.items():
    if not c: continue
    ratio = sum(n * top / total[k] for k, n in c.items()) / sum(c.values())   # độ hiếm TB (có trọng số) của ảnh
    if ratio >= MIN_RATIO:
        copies[i] = min(MAX_COPIES, int(round(ratio)) - 1)

# ---------------- GHÉP DATASET ----------------
for sub in ("images/train", "labels/train", "images/val", "labels/val"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
oi, ol = f"{OUT}/images/train", f"{OUT}/labels/train"

for i in ids:                                         # bản gốc
    link(img_path(i, SRC_IMG), f"{oi}/{i}.jpg")
    if os.path.exists(f"{SRC_LBL}/{i}.txt"): link(f"{SRC_LBL}/{i}.txt", f"{ol}/{i}.txt")

for i, n in copies.items():                           # nhân bản ảnh có tổ hợp hiếm
    for k in range(n):
        link(img_path(i, SRC_IMG), f"{oi}/{i}_rare{k}.jpg")
        link(f"{SRC_LBL}/{i}.txt", f"{ol}/{i}_rare{k}.txt")

for p in glob.glob(f"{VAL_IMG}/*.*"):                 # val thật
    n = os.path.splitext(os.path.basename(p))[0]
    link(p, f"{OUT}/images/val/{n}.jpg")
    if os.path.exists(f"{VAL_LBL}/{n}.txt"): link(f"{VAL_LBL}/{n}.txt", f"{OUT}/labels/val/{n}.txt")

with open(f"{OUT}/data.yaml", "w", encoding="utf-8") as f:
    yaml.dump({'train': f"{OUT}/images/train", 'val': f"{OUT}/images/val",
               'nc': len(CLASSES), 'names': CLASSES}, f, sort_keys=False)
for c in glob.glob(f"{OUT}/labels/*.cache"): os.remove(c)

after = Counter()
for p in glob.glob(f"{ol}/*.txt"): after += edge_combos(p)
print(f"\nẢnh train: {len(glob.glob(oi + '/*.jpg'))} (gốc {len(ids)}, ảnh được nhân: {len(copies)})")
print("Tổ hợp SAU khi nhân bản:")
for (k, m), n in sorted(after.items()):
    print(f"  {CLASSES[k]:<14}+ {m:<7}{n:>6}")
print("val ->", f"{OUT}/images/val")

Tổ hợp (kiểu nét x icon) trong train:
  edge_type_1   + oneway   2316
  edge_type_1   + stairs   3302
  edge_type_2   + oneway   1285
  edge_type_2   + stairs   1766
  edge_type_3   + oneway   1411
  edge_type_3   + stairs   1981
  print_edge_1  + oneway    740
  print_edge_1  + stairs   1121
  print_edge_2  + oneway    390
  print_edge_2  + stairs    608
  print_edge_3  + oneway    442
  print_edge_3  + stairs    646

Ảnh train: 4859 (gốc 2000, ảnh được nhân: 1564)
Tổ hợp SAU khi nhân bản:
  edge_type_1   + oneway   3869
  edge_type_1   + stairs   5147
  edge_type_2   + oneway   2487
  edge_type_2   + stairs   3185
  edge_type_3   + oneway   2685
  edge_type_3   + stairs   3528
  print_edge_1  + oneway   3485
  print_edge_1  + stairs   5106
  print_edge_2  + oneway   1929
  print_edge_2  + stairs   2921
  print_edge_3  + oneway   2171
  print_edge_3  + stairs   3096
val -> /content/YOLO_Aug/images/val


In [ ]:
import yaml
d = yaml.safe_load(open('/content/YOLO_Aug/data.yaml', encoding='utf-8'))
print(d['train']); print(d['val'])
assert d['val'].endswith('images/val') and d['val'] != d['train'], "val đang trỏ về train!"
print("OK: val là tập validation thật")

/content/YOLO_Aug/images/train
/content/YOLO_Aug/images/val
OK: val là tập validation thật


In [ ]:
import os
import shutil
from google.colab import drive
from ultralytics import YOLO

# ==========================================
# 0. KẾT NỐI VÀ CHUẨN BỊ GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')
# Backup rieng cho YOLOv8m, khong ghi de weight YOLOv8s
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/YOLO_Courier_Backup_M'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# Hàm Callback: Tự động chạy ngầm mỗi khi kết thúc 1 epoch
def backup_weights_to_drive(trainer):
    weights_dir = os.path.join(trainer.save_dir, 'weights')
    last_pt = os.path.join(weights_dir, 'last.pt')
    best_pt = os.path.join(weights_dir, 'best.pt')

    # 1. Update last.pt sau mỗi epoch
    if os.path.exists(last_pt):
        shutil.copy(last_pt, os.path.join(DRIVE_BACKUP_DIR, 'last.pt'))

    # 2. Update best.pt (YOLO chỉ ghi đè file best.pt cục bộ khi mAP tăng,
    # nên ta copy sang Drive cũng đồng nghĩa với việc Drive luôn giữ bản tốt nhất mới nhất)
    if os.path.exists(best_pt):
        shutil.copy(best_pt, os.path.join(DRIVE_BACKUP_DIR, 'best.pt'))

# ==========================================
# 1. NẠP MODEL & GẮN CALLBACK
# ==========================================
# YOLOv8m khac kien truc YOLOv8s, phai bat dau tu pretrained YOLOv8m
model = YOLO('yolov8m.pt')

# Gắn hàm backup vào sự kiện "on_fit_epoch_end"
model.add_callback("on_fit_epoch_end", backup_weights_to_drive)

# ==========================================
# 2. BẮT ĐẦU FINETUNE
# ==========================================
results = model.train(
    data="/content/YOLO_Aug/data.yaml",
    name="yolov8m_courier",

    epochs=50,
    patience=10,
    imgsz=1024,
    batch=6,               # an toan cho T4 15 GB o imgsz=1024

    optimizer="AdamW",
    lr0=0.0005,            # train transfer tu pretrained, khong phai fine-tune S
    lrf=0.1,
    cos_lr=True,
    warmup_epochs=2.0,
    freeze=0,

    cls=2.0,               # giữ
    box=6.0,               # giữ

    fliplr=0.0,            # giữ: flip làm sai class trái/phải
    flipud=0.0,
    degrees=2.0,           # cũ: 6
    translate=0.05,        # cũ: 0.15
    scale=0.40,            # cũ: 0.25
    mosaic=0.0,
    mixup=0.0,
    erasing=0.0,           # tránh xóa icon nhỏ

    resume=False,
)

# Dam bao Drive co dung best.pt cuoi cung sau khi train
BEST_MODEL_PATH = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
shutil.copy(BEST_MODEL_PATH, os.path.join(DRIVE_BACKUP_DIR, 'best.pt'))
print('YOLOv8m best:', BEST_MODEL_PATH)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=6.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/YOLO_Aug/data.yaml, degrees=2.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, 

In [ ]:
import json
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered'
]

EDGE_CLS = ('edge_type_', 'print_edge_')
ICON_CLS = {'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right'}
NODE_CLS = set(CLASSES[:14])  # landmark + robot
IMGSZ = 1024                  # PHẢI bằng kích thước lúc train


# ==========================================
# HELPERS TOA DO
# ==========================================
def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def dynamic_cluster(coords, unit_size):
    if not coords:
        return []
    coords = sorted(coords)
    clusters = [[coords[0]]]
    tolerance = unit_size * 0.4  # dung sai 40% chieu dai 1 o
    for c in coords[1:]:
        if c - np.mean(clusters[-1]) <= tolerance:
            clusters[-1].append(c)
        else:
            clusters.append([c])
    return [float(np.mean(cl)) for cl in clusters]


def get_grid_index(val, clusters):
    return min(range(len(clusters)), key=lambda i: abs(clusters[i] - val))


def is_plain_edge(c):
    return c.startswith(EDGE_CLS)


# ==========================================
# TÁCH CẠNH DÀI (split_long): 1 box phủ n ô -> n cạnh 1 ô
# ==========================================
def split_long(na, nb):
    """na, nb = [r, c] hai đầu box. Trả về list các cạnh 1 ô ((r,c),(r,c')) hoặc [] nếu chéo / trùng."""
    (r1, c1), (r2, c2) = sorted([tuple(na), tuple(nb)])
    if r1 == r2 and c1 != c2:
        return [((r1, c), (r1, c + 1)) for c in range(c1, c2)]
    if c1 == c2 and r1 != r2:
        return [((r, c1), (r + 1, c1)) for r in range(r1, r2)]
    return []


# ==========================================
# LEGEND: edge_type_k / print_edge_k -> normal | crowded | covered
# ==========================================
def build_legend(dets):
    texts = sorted([d for d in dets if d['class'].startswith('txt_')], key=lambda d: d['center'][1])
    legend = {}
    for fam, prefix in (('edge_type_', 'swatch_edge_'), ('print_edge_', 'swatch_print_')):
        sw = sorted([d for d in dets if d['class'].startswith(prefix)], key=lambda d: d['center'][1])
        if not sw or not texts:
            continue
        if len(sw) == len(texts):  # ghep theo thu tu tren xuong duoi
            pairs = list(zip(sw, texts))
        else:  # fallback: gan nhat theo y
            pairs = [(s, min(texts, key=lambda t: abs(t['center'][1] - s['center'][1]))) for s in sw]
        for s, t in pairs:
            legend[fam + s['class'][-1]] = t['class'].replace('txt_', '')
    return legend


# ==========================================
# DAC TRUNG NET VE (mau + do "lien/dut") DOC THEO 1 DOAN
# Chỉ lấy mẫu ở 18-38% và 62-82% chiều dài -> NÉ icon ở giữa cạnh
# ==========================================
def line_feature(img, a, b, bg, half=4):
    H, W = img.shape[:2]
    a, b = np.array(a, float), np.array(b, float)
    d = b - a
    L = np.linalg.norm(d)
    if L < 8:
        return None
    n = np.array([-d[1], d[0]]) / L
    ts = np.r_[np.linspace(0.18, 0.38, 25), np.linspace(0.62, 0.82, 25)]
    ink, cols = [], []
    for t in ts:
        c = a + d * t
        hit = False
        for off in range(-half, half + 1):
            x, y = np.round(c + n * off).astype(int)
            if 0 <= x < W and 0 <= y < H:
                px = img[y, x].astype(float)
                if np.linalg.norm(px - bg) > 50:
                    hit = True
                    cols.append(px)
        ink.append(hit)
    if not cols:
        return None
    return np.concatenate([np.median(cols, axis=0), [200.0 * np.mean(ink)]])


def swatch_ref_features(img, dets, legend, bg):
    """status -> feature tu o mau trong legend (ca swatch_edge va swatch_print)."""
    refs = {}
    for d in dets:
        for prefix, fam in (('swatch_edge_', 'edge_type_'), ('swatch_print_', 'print_edge_')):
            if not d['class'].startswith(prefix):
                continue
            status = legend.get(fam + d['class'][-1])
            if not status:
                continue
            x1, y1, x2, y2 = [int(v) for v in d['box']]
            px = img[max(0, y1):y2, max(0, x1):x2].reshape(-1, 3).astype(float)
            keep = np.linalg.norm(px - bg, axis=1) > 50
            if keep.sum() >= 5:
                refs[status] = np.r_[np.median(px[keep], axis=0), 200.0]
    return refs


def classify_feature(feat, centroids, default='normal'):
    if feat is None or not centroids:
        return default
    return min(centroids, key=lambda s: np.linalg.norm(feat - centroids[s]))


def _oneway_target(direction, na, nb):
    if direction == 'up':
        return [min(na[0], nb[0]), na[1]]
    if direction == 'down':
        return [max(na[0], nb[0]), na[1]]
    if direction == 'left':
        return [na[0], min(na[1], nb[1])]
    return [na[0], max(na[1], nb[1])]  # right


# ==========================================
# 1. HAM CHINH: YOLO -> JSON GRAPH
# ==========================================
def yolo_to_scene_json(image_path, model_path, mission_via=None, mission_goal=None,
                       conf=0.4, debug=False, recheck_conf=0.55):
    model = YOLO(model_path) if isinstance(model_path, str) else model_path

    # (A) Resize về 1024x1024 giống lúc train (predict gốc dùng letterbox giữ tỉ lệ -> lệch hình học)
    bgr = image_path if isinstance(image_path, np.ndarray) else cv2.imread(image_path)
    bgr = cv2.resize(bgr, (IMGSZ, IMGSZ), interpolation=cv2.INTER_AREA)
    res = model.predict(bgr, imgsz=IMGSZ, conf=conf, verbose=False)[0]
    img = bgr[:, :, ::-1]  # BGR -> RGB
    bg = np.median(img.reshape(-1, 3), axis=0)

    dets = []
    for box, cls_id, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
        dets.append({'class': CLASSES[int(cls_id)], 'box': box, 'conf': float(cf),
                     'center': get_center(box), 'w': box[2] - box[0], 'h': box[3] - box[1]})

    # --- thuoc do o luoi
    edge_like = [d for d in dets if is_plain_edge(d['class']) or d['class'] in ICON_CLS]
    hw = [d['w'] for d in edge_like if d['w'] > d['h']]
    vh = [d['h'] for d in edge_like if d['h'] > d['w']]
    grid_w = np.median(hw) if hw else 100
    grid_h = np.median(vh) if vh else 100

    def elongated(d):  # icon dai co ~1 o => icon cung la canh; icon nho => modifier
        return (d['w'] >= 0.6 * grid_w) if d['w'] >= d['h'] else (d['h'] >= 0.6 * grid_h)

    edge_dets = [d for d in dets if is_plain_edge(d['class']) or (d['class'] in ICON_CLS and elongated(d))]
    modifiers = [d for d in dets if d['class'] in ICON_CLS and not elongated(d)]

    scene = {'nodes': [], 'edges': [], 'landmarks': [], 'robot': {},
             'weather': 'rain' if any(d['class'] == 'weather_rain' for d in dets) else 'dry',
             'mission': {'via': mission_via, 'goal': mission_goal}, 'warnings': []}

    legend = build_legend(dets)
    swatch_refs = swatch_ref_features(img, dets, legend, bg)

    # --- luoi dong
    raw_x, raw_y = [], []
    for d in dets:
        if d['class'] in NODE_CLS:
            raw_x.append(d['center'][0])
            raw_y.append(d['center'][1])
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            raw_y.append(d['center'][1])
            raw_x += [x1, x2]
        else:
            raw_x.append(d['center'][0])
            raw_y += [y1, y2]
    col_cl, row_cl = dynamic_cluster(raw_x, grid_w), dynamic_cluster(raw_y, grid_h)

    # --- landmark / robot
    for d in dets:
        if d['class'] not in NODE_CLS:
            continue
        r, c = get_grid_index(d['center'][1], row_cl), get_grid_index(d['center'][0], col_cl)
        if d['class'].startswith('robot_'):
            scene['robot'] = {'rc': [r, c], 'heading': d['class'].split('_')[1].upper()}
        else:
            scene['landmarks'].append({'type': d['class'], 'rc': [r, c]})
    for r in range(len(row_cl)):
        for c in range(len(col_cl)):
            scene['nodes'].append({'rc': [r, c]})

    def endpoints(d):
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            r = get_grid_index(d['center'][1], row_cl)
            return [r, get_grid_index(x1, col_cl)], [r, get_grid_index(x2, col_cl)]
        c = get_grid_index(d['center'][0], col_cl)
        return [get_grid_index(y1, row_cl), c], [get_grid_index(y2, row_cl), c]

    def node_px(rc):
        return (col_cl[rc[1]], row_cl[rc[0]])

    # --- canh: TACH box dai thanh cac canh 1 o, gop theo cap node
    edges = {}
    for d in edge_dets:
        na, nb = endpoints(d)
        units = split_long(na, nb)
        if not units:
            if na != nb:
                scene['warnings'].append(f"{d['class']} cheo/khong hop le: {na}->{nb}")
            continue
        n_units = len(units)
        cls = d['class']
        if n_units > 1:
            scene['warnings'].append(f"{cls} phu {n_units} o: {na}->{nb} (da tach)")
            if cls in ICON_CLS:          # icon dai nhieu o: khong biet gan o nao -> xu ly nhu modifier
                modifiers.append(d)
                continue
        cf = d['conf'] * (0.5 if n_units > 1 else 1.0)   # box dai kem tin cay hon box 1 o

        for ua, ub in units:
            pa, pb = node_px(ua), node_px(ub)
            e = edges.setdefault((ua, ub), {
                'a': list(ua), 'b': list(ub), 'status': None, 'stairs': False, 'oneway_to': None,
                'yolo_center': ((pa[0] + pb[0]) / 2, (pa[1] + pb[1]) / 2),
                '_p': (pa, pb), '_closed': False, '_sconf': -1.0, '_feat': None})
            if is_plain_edge(cls):
                if cf > e['_sconf']:                       # (B) giu class kieu net co conf cao nhat
                    e['status'] = legend.get(cls, 'normal')
                    e['_sconf'] = cf
                    e['_feat'] = line_feature(img, pa, pb, bg)
            elif cls == 'closed':
                e['_closed'] = True
            elif cls == 'stairs':
                e['stairs'] = True
            elif cls.startswith('oneway_'):
                e['oneway_to'] = _oneway_target(cls.split('_')[1], list(ua), list(ub))

    # --- tam dac trung moi status (chi tu canh tin cay cao), bu bang swatch
    def build_centroids(min_conf):
        by = {}
        for e in edges.values():
            if e['status'] is not None and e['_feat'] is not None and e['_sconf'] >= min_conf:
                by.setdefault(e['status'], []).append(e['_feat'])
        return {s: np.median(v, axis=0) for s, v in by.items()}

    centroids = build_centroids(0.7) or build_centroids(0.0)
    for s, f in swatch_refs.items():
        centroids.setdefault(s, f)

    if debug:
        print("legend   :", legend)
        print("centroids:", {k: np.round(v, 1).tolist() for k, v in centroids.items()})

    for key, e in edges.items():
        # (C) canh chi co icon, KHONG co box kieu net (du phong): doan bang mau
        if e['status'] is None:
            feat = line_feature(img, e['_p'][0], e['_p'][1], bg)
            e['status'] = classify_feature(feat, centroids)
            if debug:
                print(f"  icon-edge {key}: khong co box kieu net -> mau -> {e['status']}")
        # (D) kiem tra lai canh co box kieu net nhung conf thap (vd e3 bi nham thanh e1)
        elif e['_sconf'] < recheck_conf and e['_feat'] is not None and len(centroids) >= 2:
            dist = {s: np.linalg.norm(e['_feat'] - c) for s, c in centroids.items()}
            best = min(dist, key=dist.get)
            if best != e['status'] and dist[best] < 0.7 * dist.get(e['status'], 1e9):
                if debug:
                    print(f"  recheck {key}: {e['status']}({e['_sconf']:.2f}) -> {best} (mau)")
                e['status'] = best
        if e['_closed']:
            e['status'] = 'closed'

    # --- icon nho / icon dai nhieu o -> gan vao canh gan nhat
    for m in modifiers:
        if not edges:
            break
        e = min(edges.values(), key=lambda e: get_distance(m['center'], e['yolo_center']))
        if get_distance(m['center'], e['yolo_center']) > grid_w * 0.5:
            continue
        if m['class'] == 'closed':
            e['status'] = 'closed'
        elif m['class'] == 'stairs':
            e['stairs'] = True
        else:
            e['oneway_to'] = _oneway_target(m['class'].split('_')[1], e['a'], e['b'])

    for e in edges.values():
        for k in ('yolo_center', '_p', '_closed', '_feat', '_sconf'):
            e.pop(k, None)
        scene['edges'].append(e)
    return scene

# ---- DƯỚI ĐÂY GIỮ NGUYÊN cell 14 cũ: draw_scene_map, compare_with_ground_truth, khối __main__ ----

In [ ]:
import os
import re
import json
import glob
import traceback
from collections import Counter

from tqdm import tqdm
from ultralytics import YOLO

# Can co san trong notebook: yolo_to_scene_json (ban da sua, nhan duoc object model)
# Neu chay file rieng thi: from yolo_scene_full import yolo_to_scene_json

# ==========================================
# CAU HINH
# ==========================================
val_img_dir = '/content/YOLO_Full_Dataset/images/val'
val_lbl_dir = '/content/YOLO_Full_Dataset/labels/val'   # chi la nhan box YOLO, khong dung de cham graph
SCENES_JSON = '/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/validation/scenes.json'  # SUA neu khac
MODEL_PATH = '/content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt'
CONF = 0.4
OUT_JSON = '/content/val_failures.json'


# ==========================================
# LOAD GROUND TRUTH
# ==========================================
def load_scenes(path):
    with open(path, 'r') as f:
        scenes = json.load(f)
    by_id = {}
    for i, s in enumerate(scenes):
        sid = s.get('scene_id', s.get('id', i))
        by_id[int(sid) if str(sid).isdigit() else i] = s
    return scenes, by_id


def true_scene_for(img_path, scenes, by_id):
    """val-00012.jpg -> scene so 12 (theo scene_id neu co, khong thi theo index)."""
    idx = int(re.findall(r'\d+', os.path.splitext(os.path.basename(img_path))[0])[-1])
    return by_id.get(idx, scenes[idx] if idx < len(scenes) else None)


# ==========================================
# SO SANH 1 SCENE -> TRA VE DANH SACH LOI (khong print)
# ==========================================
def diff_scene(pred, true):
    errs = []  # (loai_loi, chi_tiet)

    if pred.get('weather', 'dry') != true.get('weather', 'dry'):
        errs.append(('weather', f"pred={pred.get('weather')} true={true.get('weather')}"))

    if pred.get('robot', {}) != true.get('robot', {}):
        errs.append(('robot', f"pred={pred.get('robot')} true={true.get('robot')}"))

    lm_p = {(l['type'], tuple(l['rc'])) for l in pred.get('landmarks', [])}
    lm_t = {(l['type'], tuple(l['rc'])) for l in true.get('landmarks', [])}
    for x in sorted(lm_t - lm_p):
        errs.append(('landmark_missing', str(x)))
    for x in sorted(lm_p - lm_t):
        errs.append(('landmark_extra', str(x)))

    def edge_dict(scene):
        d = {}
        for e in scene.get('edges', []):
            a, b = sorted([tuple(e['a']), tuple(e['b'])])
            d[(a, b)] = {'status': e.get('status', 'normal'),
                         'stairs': bool(e.get('stairs', False)),
                         'oneway_to': tuple(e['oneway_to']) if e.get('oneway_to') else None}
        return d

    ep, et = edge_dict(pred), edge_dict(true)
    for k in sorted(set(et) - set(ep)):
        errs.append(('edge_missing', f"{k} true={et[k]}"))
    for k in sorted(set(ep) - set(et)):
        errs.append(('edge_extra', f"{k} pred={ep[k]}"))
    for k in sorted(set(ep) & set(et)):
        for field in ('status', 'stairs', 'oneway_to'):
            if ep[k][field] != et[k][field]:
                errs.append((f'edge_wrong_{field}', f"{k} pred={ep[k][field]} true={et[k][field]}"))
    return errs, len(et)


# ==========================================
# CHAY TOAN BO VAL
# ==========================================
def evaluate_val(verbose=True):
    scenes, by_id = load_scenes(SCENES_JSON)
    img_paths = sorted(glob.glob(os.path.join(val_img_dir, '*.jpg')) +
                       glob.glob(os.path.join(val_img_dir, '*.png')))
    print(f"So anh val: {len(img_paths)} | So scene GT: {len(scenes)}")

    model = YOLO(MODEL_PATH)  # load 1 lan
    failures, err_counter = [], Counter()
    n_ok, total_edges, total_edge_ok = 0, 0, 0

    for p in tqdm(img_paths):
        name = os.path.basename(p)
        true = true_scene_for(p, scenes, by_id)
        if true is None:
            failures.append({'image': name, 'errors': [('no_ground_truth', '')]})
            continue
        try:
            pred = yolo_to_scene_json(p, model, mission_via=true['mission'].get('via'),
                                      mission_goal=true['mission'].get('goal'), conf=CONF)
            errs, n_true_edges = diff_scene(pred, true)
        except Exception:
            failures.append({'image': name, 'errors': [('crash', traceback.format_exc(limit=2))]})
            err_counter['crash'] += 1
            continue

        total_edges += n_true_edges
        total_edge_ok += n_true_edges - sum(1 for t, _ in errs if t in ('edge_missing', 'edge_wrong_status',
                                                                        'edge_wrong_stairs', 'edge_wrong_oneway_to'))
        if errs:
            failures.append({'image': name, 'errors': errs})
            for t, _ in errs:
                err_counter[t] += 1
        else:
            n_ok += 1

    # ---------- BAO CAO ----------
    n = len(img_paths)
    print("\n" + "=" * 60)
    print(f"DUNG HOAN TOAN: {n_ok}/{n}  ({100 * n_ok / max(n, 1):.1f}%)")
    print(f"SAI: {len(failures)} anh")
    if total_edges:
        print(f"Do chinh xac canh (uoc luong): {100 * total_edge_ok / total_edges:.2f}%  ({total_edge_ok}/{total_edges})")
    print("\nThong ke loai loi:")
    for t, c in err_counter.most_common():
        print(f"  {t:25s} {c}")

    if verbose and failures:
        print("\n" + "=" * 60)
        print("DANH SACH ANH SAI:")
        for f in failures:
            print(f"\n[{f['image']}]  ({len(f['errors'])} loi)")
            for t, detail in f['errors']:
                print(f"   - {t}: {detail}")

    with open(OUT_JSON, 'w') as fp:
        json.dump(failures, fp, ensure_ascii=False, indent=1, default=str)
    print(f"\nDa luu chi tiet vao {OUT_JSON}")
    return failures


if __name__ == "__main__":
    failures = evaluate_val(verbose=True)

So anh val: 300 | So scene GT: 300


100%|██████████| 300/300 [00:35<00:00,  8.44it/s]


DUNG HOAN TOAN: 300/300  (100.0%)
SAI: 0 anh
Do chinh xac canh (uoc luong): 100.00%  (23001/23001)

Thong ke loai loi:

Da luu chi tiet vao /content/val_failures.json


In [ ]:
import json
import math
import os
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

# ==========================================
# CONFIG
# ==========================================
CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered'
]

EDGE_CLS = ('edge_type_', 'print_edge_')                  # box cạnh (kiểu nét) -> status lấy từ legend
EDGE_BOX_CLS = {'closed'}                                 # closed = box phủ cả đoạn đường
MODIFIER_CLS = {'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right'}  # icon nhỏ giữa cạnh
NODE_CLS = set(CLASSES[:14])                              # landmark + robot


# ==========================================
# HELPERS
# ==========================================
def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def dynamic_cluster(coords, unit_size):
    if not coords:
        return []
    coords = sorted(coords)
    clusters = [[coords[0]]]
    tolerance = unit_size * 0.4
    for c in coords[1:]:
        if c - np.mean(clusters[-1]) <= tolerance:
            clusters[-1].append(c)
        else:
            clusters.append([c])
    return [float(np.mean(cl)) for cl in clusters]


def get_grid_index(val, clusters):
    return min(range(len(clusters)), key=lambda i: abs(clusters[i] - val))


def is_plain_edge(c):
    return c.startswith(EDGE_CLS)


# ==========================================
# LEGEND: swatch k -> txt bên cạnh -> status
# ==========================================
def build_legend(dets, txt_side='right', pairs_out=None):
    texts = [d for d in dets if d['class'].startswith('txt_')]
    swatches = [d for d in dets if d['class'].startswith(('swatch_edge_', 'swatch_print_'))]
    if not texts or not swatches:
        return {}

    def find_label(s):
        sx1, _, sx2, _ = s['box']
        slack = 0.3 * s['w']
        row = [t for t in texts
               if abs(t['center'][1] - s['center'][1]) <= max(s['h'], t['h']) * 0.6]
        if txt_side == 'left':
            side = [t for t in row if t['box'][2] <= sx1 + slack]
        else:
            side = [t for t in row if t['box'][0] >= sx2 - slack]
        pool = side or row or texts
        return min(pool, key=lambda t: get_distance(t['center'], s['center']))

    best = {}  # k -> (conf swatch, status)
    for s in swatches:
        k = s['class'][-1]
        t = find_label(s)
        if pairs_out is not None:
            pairs_out.append((s, t))
        status = t['class'].replace('txt_', '')
        if k not in best or s['conf'] > best[k][0]:
            best[k] = (s['conf'], status)

    legend = {}
    for k, (_, status) in best.items():
        legend['edge_type_' + k] = status
        legend['print_edge_' + k] = status
    return legend


def _oneway_target(direction, na, nb):
    if direction == 'up':
        return [min(na[0], nb[0]), na[1]]
    if direction == 'down':
        return [max(na[0], nb[0]), na[1]]
    if direction == 'left':
        return [na[0], min(na[1], nb[1])]
    return [na[0], max(na[1], nb[1])]  # right


# ==========================================
# 1. YOLO -> DETS (chạy model 1 lần, lọc ngưỡng sau)
# ==========================================
def predict_dets(model, image_path, min_conf=0.1):
    res = model.predict(image_path, conf=min_conf, verbose=False)[0]
    img = res.orig_img[:, :, ::-1]  # BGR -> RGB
    dets = []
    for box, cls_id, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
        box = [float(v) for v in box]
        dets.append({'class': CLASSES[int(cls_id)], 'box': box, 'conf': float(cf),
                     'center': get_center(box), 'w': box[2] - box[0], 'h': box[3] - box[1]})
    return dets, img


def collect_dets(model, scenes, img_dir, min_conf=0.1):
    """Chạy YOLO trên toàn bộ val 1 lần, trả {scene_id: dets}. Sau đó thử ngưỡng khác không cần chạy lại model."""
    cache = {}
    for sc in tqdm(scenes, desc="YOLO predict"):
        path = os.path.join(img_dir, f"{sc['scene_id']}.jpg")
        if os.path.exists(path):
            cache[sc['scene_id']] = predict_dets(model, path, min_conf)[0]
    return cache


# ==========================================
# 2. DETS -> JSON GRAPH
#    box cạnh   : edge_type_k / print_edge_k (status = legend) / closed
#    icon nhỏ   : stairs / oneway_*  -> gán vào cạnh có box chứa tâm icon
#    split_long : box cạnh dài >1 ô -> tách thành nhiều cạnh 1 ô (False = giữ kiểu cũ: 1 cạnh nối xa)
# ==========================================
def dets_to_scene(dets, mission_via=None, mission_goal=None, conf=0.4, edge_conf=None,
                  txt_side='right', split_long=True, trace=None):
    edge_conf = conf if edge_conf is None else edge_conf
    dets = [dict(d) for d in dets
            if d['conf'] >= (edge_conf if is_plain_edge(d['class']) else conf)]

    pairs = []
    legend = build_legend(dets, txt_side=txt_side, pairs_out=pairs)
    for d in dets:
        if is_plain_edge(d['class']):
            d['status'] = legend.get(d['class'], '?')

    edge_dets = [d for d in dets if is_plain_edge(d['class']) or d['class'] in EDGE_BOX_CLS]
    modifiers = [d for d in dets if d['class'] in MODIFIER_CLS]

    # thước đo ô lưới: chỉ đo trên box cạnh (không tính icon)
    hw = [d['w'] for d in edge_dets if d['w'] > d['h']]
    vh = [d['h'] for d in edge_dets if d['h'] > d['w']]
    grid_w = float(np.median(hw)) if hw else 100.0
    grid_h = float(np.median(vh)) if vh else 100.0

    scene = {'nodes': [], 'edges': [], 'landmarks': [], 'robot': {},
             'weather': 'rain' if any(d['class'] == 'weather_rain' for d in dets) else 'dry',
             'mission': {'via': mission_via, 'goal': mission_goal}, 'warnings': []}

    # --- lưới động
    raw_x, raw_y = [], []
    for d in dets:
        if d['class'] in NODE_CLS:
            raw_x.append(d['center'][0])
            raw_y.append(d['center'][1])
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            raw_y.append(d['center'][1])
            raw_x += [x1, x2]
        else:
            raw_x.append(d['center'][0])
            raw_y += [y1, y2]
    col_cl, row_cl = dynamic_cluster(raw_x, grid_w), dynamic_cluster(raw_y, grid_h)

    # --- landmark / robot
    for d in dets:
        if d['class'] not in NODE_CLS:
            continue
        r, c = get_grid_index(d['center'][1], row_cl), get_grid_index(d['center'][0], col_cl)
        if d['class'].startswith('robot_'):
            scene['robot'] = {'rc': [r, c], 'heading': d['class'].split('_')[1].upper()}
        else:
            scene['landmarks'].append({'type': d['class'], 'rc': [r, c]})
    for r in range(len(row_cl)):
        for c in range(len(col_cl)):
            scene['nodes'].append({'rc': [r, c]})

    # --- cạnh
    edges = {}
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        horiz = d['w'] >= d['h']
        if horiz:
            r = get_grid_index(d['center'][1], row_cl)
            ca, cb = sorted([get_grid_index(x1, col_cl), get_grid_index(x2, col_cl)])
            steps = [([r, c], [r, c + 1]) for c in range(ca, cb)]
        else:
            c = get_grid_index(d['center'][0], col_cl)
            ra, rb = sorted([get_grid_index(y1, row_cl), get_grid_index(y2, row_cl)])
            steps = [([r, c], [r + 1, c]) for r in range(ra, rb)]
        if not steps:
            continue
        if len(steps) > 1:
            scene['warnings'].append(f"{d['class']} dai {len(steps)} o: {steps[0][0]}->{steps[-1][1]}"
                                     f" ({'tach' if split_long else 'giu nguyen'})")
            if not split_long:
                steps = [(steps[0][0], steps[-1][1])]

        for na, nb in steps:
            if horiz:
                box = [col_cl[na[1]], y1, col_cl[nb[1]], y2]
            else:
                box = [x1, row_cl[na[0]], x2, row_cl[nb[0]]]
            key = (tuple(na), tuple(nb))
            e = edges.setdefault(key, {'a': na, 'b': nb, 'status': None, 'stairs': False,
                                       'oneway_to': None, '_box': box, '_horiz': horiz,
                                       '_center': get_center(box), '_plain_conf': -1.0,
                                       '_closed': False, '_src': None})
            cls = d['class']
            if is_plain_edge(cls):
                if d['conf'] > e['_plain_conf']:      # 2 box plain chồng nhau -> lấy box conf cao hơn
                    e['_plain_conf'] = d['conf']
                    e['status'] = legend.get(cls, 'normal')
                    e['_src'] = ('legend:' if cls in legend else 'NO_LEGEND:') + cls
            else:  # closed
                e['_closed'] = True

    for e in edges.values():
        if e['_closed']:
            e['status'] = 'closed'
            e['_src'] = 'closed_box'

    # --- icon nhỏ -> cạnh có box chứa tâm icon, đúng hướng
    tol = 0.15 * min(grid_w, grid_h)
    for m in modifiers:
        cx, cy = m['center']
        direction = m['class'].split('_')[1] if m['class'].startswith('oneway_') else None
        best, best_d = None, None
        for e in edges.values():
            x1, y1, x2, y2 = e['_box']
            if not (x1 - tol <= cx <= x2 + tol and y1 - tol <= cy <= y2 + tol):
                continue
            if direction in ('left', 'right') and not e['_horiz']:
                continue
            if direction in ('up', 'down') and e['_horiz']:
                continue
            dist = get_distance(m['center'], e['_center'])
            if best is None or dist < best_d:
                best, best_d = e, dist
        if best is None:
            scene['warnings'].append(f"icon {m['class']} tai {np.round(m['center']).tolist()} khong nam trong canh nao")
            continue
        if m['class'] == 'stairs':
            best['stairs'] = True
        else:
            best['oneway_to'] = _oneway_target(direction, best['a'], best['b'])

    if trace is not None:
        trace.update(dets=dets, legend=legend, pairs=pairs,
                     edge_src={k: e['_src'] for k, e in edges.items()})

    for e in edges.values():
        for k in ('_box', '_horiz', '_center', '_plain_conf', '_closed', '_src'):
            e.pop(k, None)
        scene['edges'].append(e)
    return scene


def yolo_to_scene_json(image_path, model, mission_via=None, mission_goal=None, conf=0.4,
                       edge_conf=None, split_long=True, txt_side='right', trace=None):
    model = YOLO(model) if isinstance(model, str) else model
    dets, img = predict_dets(model, image_path, min_conf=min(conf, edge_conf or conf))
    scene = dets_to_scene(dets, mission_via, mission_goal, conf, edge_conf, txt_side, split_long, trace)
    if trace is not None:
        trace['img'] = img
    return scene


# ==========================================
# 3. SO SÁNH VỚI GROUND TRUTH
# ==========================================
def _edge_dict(scene):
    out = {}
    for e in scene.get('edges', []):
        k = tuple(sorted([tuple(e['a']), tuple(e['b'])]))
        ow = tuple(e['oneway_to']) if e.get('oneway_to') else None
        out[k] = {'status': e.get('status', 'normal'), 'stairs': bool(e.get('stairs')), 'oneway_to': ow}
    return out


def compare_scenes(pred, true, edge_src=None):
    """Trả (errors, counts). errors: list dict {type, key, pred, true, src}."""
    edge_src = edge_src or {}
    errors = []
    P, T = _edge_dict(pred), _edge_dict(true)

    def src_of(k):
        # edge_src key = (a, b) đã sort sẵn
        return (edge_src.get(k) or '-').split(':')[0]

    n_correct = 0
    for k in sorted(set(P) | set(T)):
        p, t = P.get(k), T.get(k)
        if p is None:
            errors.append({'type': 'edge_missing', 'key': k, 'pred': None, 'true': t, 'src': '-'})
        elif t is None:
            errors.append({'type': 'edge_extra', 'key': k, 'pred': p, 'true': None, 'src': src_of(k)})
        else:
            ok = True
            for field, etype in (('status', 'edge_wrong_status'), ('stairs', 'edge_wrong_stairs'),
                                 ('oneway_to', 'edge_wrong_oneway_to')):
                if p[field] != t[field]:
                    ok = False
                    errors.append({'type': etype, 'key': k, 'pred': p[field], 'true': t[field],
                                   'src': src_of(k)})
            n_correct += ok

    lm_p = {(lm['type'], tuple(lm['rc'])) for lm in pred.get('landmarks', [])}
    lm_t = {(lm['type'], tuple(lm['rc'])) for lm in true.get('landmarks', [])}
    for x in sorted(lm_t - lm_p):
        errors.append({'type': 'landmark_missing', 'key': x, 'pred': None, 'true': x, 'src': '-'})
    for x in sorted(lm_p - lm_t):
        errors.append({'type': 'landmark_extra', 'key': x, 'pred': x, 'true': None, 'src': '-'})

    def rob(sc):
        r = sc.get('robot') or {}
        return (tuple(r['rc']), r.get('heading', '').upper()) if r.get('rc') else None
    if rob(pred) != rob(true):
        errors.append({'type': 'robot_wrong', 'key': 'robot', 'pred': rob(pred), 'true': rob(true), 'src': '-'})

    def wea(sc):
        return 'dry' if sc.get('weather', 'dry') == 'dry' else 'rain'
    if wea(pred) != wea(true):
        errors.append({'type': 'weather_wrong', 'key': 'weather', 'pred': wea(pred), 'true': wea(true), 'src': '-'})

    return errors, {'gt_edges': len(T), 'pred_edges': len(P), 'edge_correct': n_correct}


# ==========================================
# 4. QUÉT TOÀN BỘ VAL
# ==========================================
def evaluate_val(scenes, dets_cache, conf=0.4, edge_conf=None, split_long=True, txt_side='right',
                 quiet=False, max_list=80, save_path=None):
    n_img = n_exact = 0
    tot = Counter()
    err_types = Counter()
    status_conf = Counter()      # (pred -> true) cho edge_wrong_status
    status_src = Counter()       # nguồn status của cạnh bị sai status
    failures = {}

    for sc in scenes:
        sid = sc['scene_id']
        if sid not in dets_cache:
            continue
        trace = {}
        pred = dets_to_scene(dets_cache[sid], sc['mission'].get('via'), sc['mission'].get('goal'),
                             conf=conf, edge_conf=edge_conf, txt_side=txt_side,
                             split_long=split_long, trace=trace)
        errs, cnt = compare_scenes(pred, sc, trace['edge_src'])
        n_img += 1
        tot.update(cnt)
        if not errs:
            n_exact += 1
            continue
        for e in errs:
            err_types[e['type']] += 1
            if e['type'] == 'edge_wrong_status':
                status_conf[(e['pred'], e['true'])] += 1
                status_src[e['src']] += 1
        failures[sid] = [{**e, 'key': str(e['key']), 'pred': str(e['pred']), 'true': str(e['true'])}
                         for e in errs]

    rec = tot['edge_correct'] / max(tot['gt_edges'], 1)
    prec = tot['edge_correct'] / max(tot['pred_edges'], 1)
    summary = {'images': n_img, 'exact': n_exact, 'exact_rate': n_exact / max(n_img, 1),
               'edge_recall': rec, 'edge_precision': prec, 'errors': dict(err_types)}

    if not quiet:
        print("=" * 64)
        print(f"conf={conf} edge_conf={edge_conf} split_long={split_long}")
        print(f"DUNG HOAN TOAN : {n_exact}/{n_img} anh ({100 * n_exact / max(n_img, 1):.1f}%)  | SAI: {n_img - n_exact} anh")
        print(f"Canh dung nguyen ven: {tot['edge_correct']}/{tot['gt_edges']} GT "
              f"(recall {100 * rec:.2f}%) | {tot['pred_edges']} canh du doan (precision {100 * prec:.2f}%)")
        print("\nLoai loi:")
        for k, v in err_types.most_common():
            print(f"  {k:<24}{v}")
        if status_conf:
            print("\nedge_wrong_status  (pred -> true):")
            for (p, t), v in status_conf.most_common():
                print(f"  {p:>8} -> {t:<8} {v}")
            print("nguon status cua cac canh sai (NO_LEGEND = thieu legend, legend = model phan loai sai net):")
            for k, v in status_src.most_common():
                print(f"  {k:<14}{v}")
        print("\nDANH SACH ANH SAI:")
        for sid in sorted(failures)[:max_list]:
            print(f"[{sid}] ({len(failures[sid])} loi)")
            for e in failures[sid]:
                print(f"   - {e['type']}: {e['key']} pred={e['pred']} true={e['true']}")
        if len(failures) > max_list:
            print(f"... con {len(failures) - max_list} anh nua (xem file json)")
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(failures, f, ensure_ascii=False, indent=1)
        if not quiet:
            print(f"\nDa luu chi tiet vao {save_path}")
    return summary


def sweep(scenes, dets_cache, configs):
    """configs: list (ten, dict kwargs cho evaluate_val). Chi hau xu ly, khong chay lai YOLO."""
    print(f"{'config':<28}{'exact':>10}{'recall':>9}{'prec':>9}   loi")
    for name, kw in configs:
        s = evaluate_val(scenes, dets_cache, quiet=True, **kw)
        n_err = sum(s['errors'].values())
        print(f"{name:<28}{s['exact']:>4}/{s['images']:<5}{100 * s['edge_recall']:>8.2f}%"
              f"{100 * s['edge_precision']:>8.2f}%   {n_err}")


# ==========================================
# 5. VẼ (debug 1 ảnh)
# ==========================================
def show_labeled_detections(trace, figsize=(18, 18)):
    COL = {'normal': 'limegreen', 'crowded': 'red', 'covered': 'blue', '?': 'magenta'}
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(trace['img'])
    for d in trace['dets']:
        cls = d['class']
        x1, y1, x2, y2 = d['box']
        if is_plain_edge(cls):
            st = d.get('status', '?')
            color = COL.get(st, 'magenta')
            tag = cls.replace('print_edge_', 'p').replace('edge_type_', 'e') + '=' + st
            tx, ty, va = (x1 + x2) / 2, (y1 + y2) / 2, 'center'
        elif cls.startswith(('swatch_', 'txt_')):
            color, tag, tx, ty, va = 'red', f"{cls} {d['conf']:.2f}", x1, y1 - 2, 'bottom'
        elif cls in MODIFIER_CLS or cls == 'closed':
            color, tag, tx, ty, va = 'darkorange', cls, x1, y1 - 2, 'bottom'
        elif cls.startswith('weather_'):
            color, tag, tx, ty, va = 'gray', cls, x1, y1 - 2, 'bottom'
        else:
            color, tag, tx, ty, va = 'cyan', cls, x1, y1 - 2, 'bottom'
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.3))
        ax.text(tx, ty, tag, color=color, fontsize=7, ha='center' if va == 'center' else 'left', va=va,
                fontweight='bold', bbox=dict(facecolor='white', alpha=0.65, pad=0.5, edgecolor='none'))
    for s, t in trace['pairs']:
        ax.annotate('', xy=t['center'], xytext=s['center'], arrowprops=dict(arrowstyle='->', color='lime', lw=2))
    ax.set_title("DETECTION + LABEL (e/p = edge_type/print_edge, sau dau '=' la status theo legend)")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def draw_scene_map(scene, title="Scene Map", ax=None, highlight=None):
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(title, fontsize=14, fontweight='bold')
    highlight = highlight or set()

    def get_xy(rc):
        return rc[1], -rc[0]

    color_map = {'normal': 'black', 'crowded': 'red', 'covered': 'blue', 'closed': 'gray'}
    style_map = {'normal': '-', 'crowded': '--', 'covered': ':', 'closed': '-.'}

    for edge in scene.get('edges', []):
        x1, y1 = get_xy(edge['a'])
        x2, y2 = get_xy(edge['b'])
        st = edge.get('status', 'normal')
        ax.plot([x1, x2], [y1, y2], color=color_map.get(st, 'black'),
                linestyle=style_map.get(st, '-'), linewidth=3, alpha=0.7, zorder=1)
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        key = tuple(sorted([tuple(edge['a']), tuple(edge['b'])]))
        if key in highlight:
            ax.plot([x1, x2], [y1, y2], color='magenta', linewidth=11, alpha=0.3, zorder=0)
            ax.text(mx, my + 0.12, st, color='magenta', fontsize=9, ha='center', fontweight='bold', zorder=6)
        if st == 'closed':
            ax.text(mx, my, 'X', color='darkred', fontsize=16, fontweight='bold', ha='center', va='center', zorder=3)
        if edge.get('stairs'):
            ax.text(mx, my, '|||', color='orange', fontsize=14, fontweight='bold', ha='center', va='center', zorder=3)
        if edge.get('oneway_to'):
            tx, ty = get_xy(edge['oneway_to'])
            other = edge['a'] if tuple(edge['oneway_to']) == tuple(edge['b']) else edge['b']
            ox, oy = get_xy(other)
            ux, uy = (tx - ox), (ty - oy)
            ax.annotate('', xy=(mx + ux * 0.2, my + uy * 0.2), xytext=(mx - ux * 0.2, my - uy * 0.2),
                        arrowprops=dict(arrowstyle='-|>', color='green', lw=3), zorder=4)

    for lm in scene.get('landmarks', []):
        x, y = get_xy(lm['rc'])
        ax.scatter(x, y, s=600, c='lightgreen', edgecolors='green', zorder=4)
        ax.text(x, y + 0.25, lm['type'], fontsize=11, ha='center', fontweight='bold', zorder=5)

    if scene.get('robot'):
        rx, ry = get_xy(scene['robot']['rc'])
        heading = scene['robot'].get('heading', 'UP')
        dx, dy = {'UP': (0, 0.4), 'DOWN': (0, -0.4), 'LEFT': (-0.4, 0), 'RIGHT': (0.4, 0)}.get(heading, (0, 0))
        ax.scatter(rx, ry, s=800, c='cyan', edgecolors='blue', marker='o', zorder=6)
        ax.arrow(rx, ry, dx, dy, head_width=0.2, head_length=0.2, fc='blue', ec='blue', zorder=7)
        ax.text(rx, ry - 0.3, "ROBOT", fontsize=11, color='blue', ha='center', fontweight='bold')

    ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_aspect('equal')
    if own:
        plt.show()


def diagnose(image_path, model, true_scene, conf=0.4, edge_conf=None, split_long=True,
             txt_side='right', show=True):
    trace = {}
    pred = yolo_to_scene_json(image_path, model, true_scene['mission'].get('via'),
                              true_scene['mission'].get('goal'), conf=conf, edge_conf=edge_conf,
                              split_long=split_long, txt_side=txt_side, trace=trace)
    errs, _ = compare_scenes(pred, true_scene, trace['edge_src'])
    print("=" * 70)
    print(image_path)
    print("legend:", trace['legend'])
    for s, t in trace['pairs']:
        print(f"  {s['class']} ({s['conf']:.2f}) -> {t['class']} ({t['conf']:.2f})")
    for e in errs:
        print(f"  {e['type']:<22}{e['key']} pred={e['pred']} true={e['true']} src={e['src']}")
    if not errs:
        print("  => DUNG HOAN TOAN")
    if pred['warnings']:
        print("warnings:", pred['warnings'])
    if show:
        wrong = {e['key'] for e in errs if isinstance(e['key'], tuple) and len(e['key']) == 2
                 and isinstance(e['key'][0], tuple)}
        show_labeled_detections(trace)
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        draw_scene_map(true_scene, "GROUND TRUTH", ax=axs[0], highlight=wrong)
        draw_scene_map(pred, "DU DOAN (hong = canh sai)", ax=axs[1], highlight=wrong)
        plt.tight_layout()
        plt.show()
    return pred, trace, errs


# ==========================================
# 6. CHẠY TRÊN COLAB
# ==========================================
MODEL_PATH = '/content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt'
IMG_DIR = '/content/YOLO_Full_Dataset/images/val/'
SCENES_PATH = '/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/validation/scenes.json'

MODEL = YOLO(MODEL_PATH)
with open(SCENES_PATH, 'r') as f:
    SCENES = json.load(f)

# chạy YOLO 1 lần trên toàn bộ val (min_conf thấp để còn thử edge_conf nhỏ)
DETS = collect_dets(MODEL, SCENES, IMG_DIR, min_conf=0.1)

# báo cáo chi tiết với cấu hình mặc định
evaluate_val(SCENES, DETS, conf=0.4, edge_conf=None, split_long=True, save_path='/content/val_failures.json')

# so sánh nhanh vài cấu hình hậu xử lý (không chạy lại model)
sweep(SCENES, DETS, [
    ('merge (kieu cu)',           dict(conf=0.4, split_long=False)),
    ('split_long',                dict(conf=0.4, split_long=True)),
    ('split + edge_conf 0.25',    dict(conf=0.4, edge_conf=0.25, split_long=True)),
    ('split + edge_conf 0.15',    dict(conf=0.4, edge_conf=0.15, split_long=True)),
    ('split + conf 0.3',          dict(conf=0.4, split_long=True)),
    ('split + conf 0.5',          dict(conf=0.5, split_long=True)),
])

# soi 1 ảnh cụ thể (bỏ comment):
# i = next(k for k, s in enumerate(SCENES) if s['scene_id'] == 'validation-00017')
# diagnose(IMG_DIR + 'validation-00017.jpg', MODEL, SCENES[i])

YOLO predict:   0%|          | 0/300 [00:00<?, ?it/s]

conf=0.4 edge_conf=None split_long=True
DUNG HOAN TOAN : 300/300 anh (100.0%)  | SAI: 0 anh
Canh dung nguyen ven: 23001/23001 GT (recall 100.00%) | 23001 canh du doan (precision 100.00%)

Loai loi:

DANH SACH ANH SAI:

Da luu chi tiet vao /content/val_failures.json
config                           exact   recall     prec   loi
merge (kieu cu)              297/300     99.98%   99.98%   8
split_long                   300/300    100.00%  100.00%   0
split + edge_conf 0.25       300/300    100.00%  100.00%   0
split + edge_conf 0.15       300/300    100.00%  100.00%   0
split + conf 0.3             300/300    100.00%  100.00%   0
split + conf 0.5             300/300    100.00%  100.00%   0


In [ ]:
import json
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/YOLO_Courier_Backup_M/best.pt"
IMG_DIR = "/content/YOLO_Full_Dataset/images/val"
SCENES_PATH = "/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public/validation/scenes.json"

model = YOLO(MODEL_PATH)

with open(SCENES_PATH, encoding="utf-8") as f:
    scenes = json.load(f)

# YOLO chỉ chạy một lần
dets = collect_dets(model, scenes, IMG_DIR, min_conf=0.1)

# In toàn bộ lỗi
summary = evaluate_val(
    scenes,
    dets,
    conf=0.4,
    edge_conf=None,
    split_long=True,
    max_list=300,
    save_path="/content/val_failures_finetune.json",
)

print("\nSUMMARY:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

YOLO predict:   0%|          | 0/300 [00:00<?, ?it/s]

conf=0.4 edge_conf=None split_long=True
DUNG HOAN TOAN : 300/300 anh (100.0%)  | SAI: 0 anh
Canh dung nguyen ven: 23001/23001 GT (recall 100.00%) | 23001 canh du doan (precision 100.00%)

Loai loi:

DANH SACH ANH SAI:

Da luu chi tiet vao /content/val_failures_finetune.json

SUMMARY:
{
  "images": 300,
  "exact": 300,
  "exact_rate": 1.0,
  "edge_recall": 1.0,
  "edge_precision": 1.0,
  "errors": {}
}


In [ ]:
import os
import json

FAIL_JSON = "/content/val_failures_finetune.json"

with open(FAIL_JSON, encoding="utf-8") as f:
    failures = json.load(f)

scene_by_id = {str(s["scene_id"]): s for s in scenes}

for sid in failures:
    scene = scene_by_id[str(sid)]
    image_path = os.path.join(IMG_DIR, f"{scene['scene_id']}.jpg")

    diagnose(
        image_path=image_path,
        model=model,
        true_scene=scene,
        conf=0.6,
        edge_conf=None,
        split_long=True,
        show=True,
    )

Éo còn ảnh lỗi để in =)))